# 06 · 4-bit, and the clever NF4 grid

**Recap of notebook 05:** quantization = round each number to the nearest *allowed*
value; the gap is the error. **4 bits → only 16 allowed values**, and we used an
**evenly-spaced** grid, which was rough.

This notebook fixes the roughness with **NF4**, compares it to its cousin **FP4**,
then shows the storage tricks — **scales**, **blocks**, and **double quantization** —
that keep 4-bit cheap.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## Step 1 — Neural-net weights cluster near zero

Trained weights aren't spread evenly between their smallest and largest value. They
pile up **near zero**, with only a few large ones — a bell shape (a *normal*
distribution). Let's confirm:

In [ ]:
w = rng.normal(0, 1, size=100_000)        # weights, bell-shaped around 0

print(f"within ±1 of zero : {np.mean(np.abs(w) < 1):.0%} of all weights")
print(f"within ±2 of zero : {np.mean(np.abs(w) < 2):.0%}")
print(f"out past ±3       : {np.mean(np.abs(w) > 3):.1%}  (almost none)")

So **most weights live in a narrow band near zero**, and the far-out values are
nearly empty.

## Step 2 — Why an evenly-spaced grid is wasteful

An evenly-spaced 16-value grid spends the *same* number of allowed values in the
empty tails as it does near zero where almost everything lives. We'd rather pack
**many values near zero** (used constantly) and **few in the tails** (rarely needed).

## Step 3 — NF4: place the 16 values to match the bell shape

**NF4 ("Normal Float, 4-bit")** positions its 16 values using the *quantiles* of a
normal distribution — dense near zero, sparse in the tails. Build both grids,
quantize the same bell-shaped weights, and watch NF4 win at the **same 4 bits**:

In [ ]:
w = rng.normal(0, 1, size=100_000)
w = w / np.abs(w).max()                       # normalize to [-1, 1]

uniform = np.linspace(-1, 1, 16)              # 16 evenly-spaced values
nf4     = np.quantile(w, np.linspace(0, 1, 16))   # 16 values placed by the bell shape

def quantize(w, grid):
    idx = np.abs(w[:, None] - grid[None, :]).argmin(axis=1)
    return grid[idx]

err_uniform = np.abs(w - quantize(w, uniform)).mean()
err_nf4     = np.abs(w - quantize(w, nf4)).mean()
print(f"evenly-spaced grid : mean |error| = {err_uniform:.4f}")
print(f"NF4-style grid     : mean |error| = {err_nf4:.4f}")
print(f"NF4 is {err_uniform/err_nf4:.1f}x more accurate, at the SAME 4 bits")

## Step 4 — NF4 vs FP4: two *different* non-uniform grids

You'll hear **FP4** offered as the alternative to NF4, often with a misconception:
that **FP4 is the evenly-spaced grid.** It isn't. The evenly-spaced grid is the
**uniform / INT4** one from Step 2. FP4 is something else:

- **FP4 = 4-bit *floating point*.** Its 16 values come from the float format itself —
  sign + exponent + mantissa bits. Floating-point numbers are naturally **denser near
  zero**, sparser far out. But it's a **fixed numeric format** — it knows nothing
  about your actual weights.
- **NF4** places its values at the **quantiles of a normal distribution** — it is
  **data-distribution-aware** (it assumes weights are bell-shaped), so it hugs zero
  the tightest.

Both are non-uniform and denser near zero; they differ in **why** the values sit
where they do. Let's line all three up:

In [ ]:
uniform = np.linspace(-1, 1, 16)                             # evenly spaced (INT4)

fp4_mag = np.array([0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])  # the 8 positive FP4 levels
fp4 = np.unique(np.concatenate([-fp4_mag, fp4_mag]))
fp4 = fp4 / np.abs(fp4).max()

nf4 = np.quantile(rng.normal(0, 1, 500_000), np.linspace(0, 1, 16))
nf4 = nf4 / np.abs(nf4).max()

print("positive-half levels (notice the spacing):")
for name, g in [("uniform", uniform), ("fp4", fp4), ("nf4", nf4)]:
    print(f"  {name:>7}:", np.round(g[g >= 0], 3))

w = rng.normal(0, 1, 200_000)
w = w / np.abs(w).max()
def err(grid):
    idx = np.abs(w[:, None] - grid[None, :]).argmin(axis=1)
    return np.abs(w - grid[idx]).mean()

print("\nmean error on normal weights:")
for name, g in [("uniform", uniform), ("fp4", fp4), ("nf4", nf4)]:
    print(f"  {name:>7}: {err(g):.4f}")

Uniform clearly loses; **fp4 and nf4 are both non-uniform and much better**, and nf4's
levels hug zero the tightest. In this toy demo fp4 and nf4 come out close — I'm being
honest that it doesn't cleanly separate them. The real evidence is the **QLoRA paper**,
which found **nf4 slightly but consistently beats fp4** on actual LLMs, which is why
QLoRA defaults to nf4.

**This is an exposed knob in the lab:** `bnb_4bit_quant_type` in `model_factory.py`,
currently `"nf4"` — you can flip it to `"fp4"` and measure the difference on the SQL
task (everything else identical). Unlike block size, this one you *can* sweep.

*(Footnote: the FP4 values above are the standard OCP **E2M1** set, used here to show
the *shape* — non-uniform, denser near zero. `bitsandbytes`' actual FP4 uses its own
specific code values that differ slightly; the placement idea and the takeaway are the
same.)*

## Step 5 — The "scale": fitting real weights onto a fixed grid

The NF4 grid is **fixed between −1 and +1**. But real weights aren't in that range —
one group might span ±0.05, another ±3. So before snapping, we **squeeze the weights
into [−1, 1]** by dividing by their biggest magnitude. That divisor is the **scale**.

By hand, for `[0.2, −0.8, 0.4, 0.1]`: the biggest magnitude is `0.8`, so the scale is
`0.8`. Divide by it to quantize, **multiply by it to recover** — so the scale must be
*stored*.

In [ ]:
block = np.array([0.2, -0.8, 0.4, 0.1])

scale      = np.abs(block).max()      # the scale = biggest magnitude
normalized = block / scale            # now inside [-1, 1]
recovered  = normalized * scale       # multiply back by the stored scale

print("scale (must be stored) :", scale)
print("normalized into [-1,1] :", normalized)
print("recovered (= original) :", recovered)

## Step 6 — Why split into *blocks*? (outliers)

Why not store **one** scale for the whole weight matrix? Because the scale is the
**biggest magnitude** — so a single **outlier** weight hijacks it. If one weight is
`9.0` while the rest are ~`0.1`, the global scale becomes `9.0`. Every normal weight,
divided by `9.0`, collapses to ~`0.01` and rounds to nearly the same grid value.
**One outlier wrecks the precision of all the others.** The fix: cut the weights into
small **blocks**, each with its **own** scale, so an outlier only spoils its own block.

In [ ]:
w = rng.normal(0, 0.1, size=256)     # mostly small weights...
w[100] = 9.0                          # ...with a couple of big OUTLIERS
w[200] = -7.0
grid = np.linspace(-1, 1, 16)         # (the grid's shape doesn't matter for this point)

def q_block(b):
    s = np.abs(b).max()
    if s == 0:
        return b.copy()
    idx = np.abs((b / s)[:, None] - grid).argmin(axis=1)
    return grid[idx] * s

one_scale = q_block(w)                # (a) ONE scale for the whole tensor

size = 64                             # (b) a separate scale per block of 64
per_block = w.copy()
for i in range(0, len(w), size):
    per_block[i:i+size] = q_block(w[i:i+size])

small = np.abs(w) < 1.0
print(f"global scale forced to {np.abs(w).max():.0f} (set by the outlier)")
print(f"mean error on normal weights, ONE global scale : {np.abs(w-one_scale)[small].mean():.4f}")
print(f"mean error on normal weights, per-block scales : {np.abs(w-per_block)[small].mean():.4f}")

It's a **tradeoff**: one scale is cheapest but outliers ruin everything; tiny blocks
give the best precision but you store *a lot* of scales. QLoRA picks **blocks of 64**
as the sweet spot.

## Step 7 — Double quantization: quantize the scales too

Blocking means one 32-bit scale **per 64 weights** = `32/64 = 0.5` extra bits on every
weight. But a model has **millions** of scales — just numbers — so we quantize *them*
too, to **8-bit**. Quantizing twice (weights, then scales) is **double quantization**:

In [ ]:
block_size = 64
plain  = 32 / block_size                          # one 32-bit scale per 64 weights
double = 8 / block_size + 32 / (block_size * 256) # scales -> 8-bit (+ tiny 2nd scale)

print(f"overhead WITHOUT double-quant : {plain:.3f} bits per weight")
print(f"overhead WITH    double-quant : {double:.3f} bits per weight")
print(f"saved                         : {plain - double:.3f} bits per weight")

## Recap

- Weights **cluster near zero**, so evenly-spaced (uniform/INT4) values waste precision.
- **NF4** places its 16 values by the bell shape (data-aware); **FP4** is also
  non-uniform but placed by the float format (not data-aware) — nf4 wins slightly,
  and it's the `bnb_4bit_quant_type` knob you can flip.
- A **scale** squeezes weights onto the fixed [−1, 1] grid; stored **per block** so one
  **outlier** can't wreck everyone (blocks of 64 = the sweet spot).
- **Double quantization** compresses those scales (32→8 bit), ~0.5 → ~0.13 bits/weight.

**BAM!** Next: **`07 — QLoRA, assembled`.**